# LOCUS — ARC-AGI Competition Notebook

Runs the LOCUS companion agent in competition mode against ARC-AGI games.
Scripts and companion file are fetched directly from GitHub — no dataset upload needed.

**Before running:**
1. Enable *internet* in notebook → Settings → Environment
2. Add `ANTHROPIC_API_KEY` and `ARC_API_KEY` under *Settings → Add-ons → Secrets* and toggle each on

In [ ]:
# Install dependencies
!pip install anthropic arc-agi -q

In [ ]:
# Inject Kaggle secrets into environment
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")
os.environ["ARC_API_KEY"]       = secrets.get_secret("ARC_API_KEY")

In [ ]:
# Download scripts from GitHub
BASE = "https://raw.githubusercontent.com/antfriend/companion_arc/main"
!wget -q {BASE}/kaggle_agent.py
!wget -q {BASE}/launch_competition.py

In [ ]:
# Run competition attempt
# Companion is fetched from GitHub automatically (launch_competition.py falls back to COMPANION_URL)
# Session log is written to /kaggle/working/session_log_ls20.md — download and apply locally
!python launch_competition.py ls20

## After a run: updating companion_arcprize.md

`launch_competition.py` writes `/kaggle/working/session_log_ls20.md` — the LOCUS-generated
session record and `[ew]` metadata updates in TTDB format.

**To apply the learnings locally:**

1. Download `session_log_ls20.md` from the Output tab
2. In Claude Code, run `@locus dream` — it will read the log and write the updates into `companion_arcprize.md`
3. Commit and push; next notebook run picks up the updated companion via GitHub

**What LOCUS logs during a run:**

| Event | LOCUS call | Purpose |
|---|---|---|
| Session start | `FOCUS lat-10lon10` | Increment sal on game state record |
| Session start | `STATUS` | EPS scan — surface high-strain beliefs |
| Each step | state + frame | Action decision with mechanic reasoning |
| Level win | `LOG level N complete` | Record step count + frame state |
| Level win | revision cycle prompt | Phases 1-3: notice → encounter → revise |
| Game end | `LOG win/game_over` | Final outcome with frame summary |

The revision cycle's **Phase 4** (validate) fires the next run — confirming or refuting whatever was revised.